provare mse e loss paper o altre loss da reference
Implementare: parametri pretrained, mettere autocast float16 o 32, dimensioni output e canali ragionate

prima optuno il modello poi faccio training possibilmente kfold

guida per costruire convautoencoder riguardare anche i notebook dei lab

https://apxml.com/courses/applied-autoencoders-feature-extraction/chapter-5-convolutional-autoencoders-image-data/upsampling-techniques-decoders
https://www.digitalocean.com/community/tutorials/convolutional-autoencoder
https://www.digitalocean.com/community/tutorials/writing-cnns-from-scratch-in-pytorch

In [1]:
import numpy as np
import optuna
import random
import time
from PIL import Image
from pathlib import Path

#pytorch stuff
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.models import resnet18, ResNet18_Weights

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    print("Running on GPU")
else:
    device = torch.device('cpu')
    print("Running on CPU")

## Load the dataset

In [ ]:
#### DATASET PREPROCESSING #####
img_transf = transforms.Compose([transforms.Resize((480,640)), transforms.ToTensor(), transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
map_transf = transforms.Compose([transforms.Resize((480,640)), transforms.ToTensor()]) #transforms.Normalize((0.5,), (0.5,))])

In [ ]:
#### CREATE DATASET ####
class create_dataset(Dataset):
    """ Custom dataset for loading input images and their optional ground-truth saliency maps. """
    def __init__(self, image_paths, map_paths=None, image_transform=img_transf, map_transform=map_transf):
        self.image_paths = image_paths
        self.map_paths = map_paths
        self.image_transform = image_transform
        self.map_transform = map_transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        
        image = Image.open(self.image_paths[idx]).convert("RGB")

        if self.image_transform is not None:
            image = self.image_transform(image)

        # no groundtruth
        if self.map_paths is None:
            return image

        saliency_map = Image.open(self.map_paths[idx]).convert("L")

        if self.map_transform is not None:
            saliency_map = self.map_transform(saliency_map)

        return image, saliency_map

def create_path(path):
    """ This function return a sorted list of path objects contained in the given directory. """
    return sorted(Path(path).iterdir())

In [ ]:
#### PATHS ####
train_images_path = create_path("dataset/images/train/")
train_maps_path = create_path("dataset/maps/train/")

val_images_path = create_path("dataset/images/val/")
val_maps_path = create_path("dataset/maps/val/")

In [ ]:
#training set 10k
training_set = create_dataset(train_images_path, train_maps_path)

#validation set 5k
full_validation_set = create_dataset(val_images_path, val_maps_path)

#seed for repeating the same experiment with all models, namely the same sets
rng = np.random.default_rng(42)
indices = rng.permutation(len(full_validation_set))
split = int(len(indices) * 0.6)

#3k validation
validation_set = Subset(full_validation_set, indices[:split].tolist())

#2k test
test_set = Subset(full_validation_set, indices[split:].tolist())

## Model

In [ ]:
class resnet18_encoder(nn.Module):
    """ Encoder class based on resnet18.
        The encoder can be frozen by setting frozen=True.
        Pretrained ImageNet weights can be used by setting pretrained=True. """
    def __init__(self, pretrained=False, frozen=False):
        super().__init__()

        #weights choice
        if pretrained:
            weights = ResNet18_Weights.IMAGENET1K_V1
        else:
            weights = None

        resnet = resnet18(weights=weights)

        #initial layer
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool

        #residual layers
        self.layer1 = resnet.layer1    # H/4 * W/4
        self.layer2 = resnet.layer2    # H/8 * W/8
        self.layer3 = resnet.layer3    # H/16 * W/16
        self.layer4 = resnet.layer4    # H/16 * W/16 (originally was /32, but i set the stride 1)
        self.layer4[0].conv1.stride = (1, 1) # set stride 1 instead of 2
        self.layer4[0].downsample[0].stride = (1, 1) # skip connection with stride 1 instead of 2

        self.frozen = frozen #frozen state for the encoder
        if self.frozen:
            self.freeze()

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return x

    def freeze(self):
        """ Freeze all encoder parameters and set encoder in eval mode. """
        self.frozen = True
        for p in self.parameters():
            p.requires_grad = False
        self.eval()

    def unfreeze(self):
        """ unfreeze encoder parameters."""
        self.frozen = False
        for p in self.parameters():
            p.requires_grad = True
        self.train()

    def train(self, mode=True):
        """ Keep the encoder in eval mode when is frozen to prevent BN updates."""
        super().train(mode)
        if self.frozen:
            super().train(False)
        return self

In [ ]:
class decoder(nn.Module):
    def __init__(self):
        super().__init__()

        # upsampling to the original resolution
        self.up_layer1 = nn.Sequential(nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),nn.BatchNorm2d(256), nn.ReLU(inplace=True))
        self.up_layer2 = nn.Sequential(nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.up_layer3 = nn.Sequential(nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.up_layer4 = nn.Sequential(nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2,padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        #pxlwise saliency prediction
        self.classifier = nn.Conv2d(in_channels=32, out_channels=1, kernel_size=1)


    def forward(self, x):
        x = self.up_layer1(x)
        x = self.up_layer2(x)
        x = self.up_layer3(x)
        x = self.up_layer4(x)
        return self.classifier(x)

In [ ]:
class encoder_decoder(nn.Module):
    def __init__(self, pretrained=False, frozen=False):
        super().__init__()
        self.encoder = resnet18_encoder(pretrained=pretrained, frozen=frozen)
        self.decoder = decoder()

    def forward(self, x):
        # skip gradient tracking when encoder is frozen
        if self.encoder.frozen:
            with torch.no_grad():
                features = self.encoder(x)
        else:
            features = self.encoder(x)

        logits = self.decoder(features)

        return logits

    def freeze_encoder(self):
        self.encoder.freeze()

    def unfreeze_encoder(self):
        self.encoder.unfreeze()

In [ ]:
#### SANITY CHECK ####

#img, map = training_set[0]
#
#print("train samples:", len(training_set))
#print("validation samples:", len(validation_set))
#print("test samples:", len(test_set))
#print("image shape:", img.shape)
#print("map shape:", map.shape)
#
#test_model = resnet18_encoder(pretrained=False, frozen=False).to(device)
#
#with torch.no_grad():
#    output = test_model(img.unsqueeze(0).to(device))
#
#print("output shape:", output.shape)
#
#del test_model

## Optuna objective and training

In [ ]:
class trainer:
    """ trainer class for model training, validation, and best model selection. """
    def __init__(self, model, optimizer, loss_function, device):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.loss_function = loss_function
        self.device = device
        self.use_amp = self.device.type == "cuda" #automatic mixed precision on cuda devices
        self.scaler = torch.amp.GradScaler("cuda", enabled=self.use_amp)

    def train_epoch(self, train_loader):
        self.model.train()
        total_loss = 0
        total_samples = 0

        for images, maps in train_loader:
            images = images.to(self.device, non_blocking=True)
            maps = maps.to(self.device, non_blocking=True)
            self.optimizer.zero_grad(set_to_none=True) #delete gradietns from previous batch

            with torch.autocast(device_type=self.device.type, dtype=torch.float16, enabled=self.use_amp):
                logits = self.model(images)
                loss = self.loss_function(logits, maps)

            self.scaler.scale(loss).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()
            
            batch_size = images.size(0)
            total_loss += (loss.detach().float().item() * batch_size)
            total_samples += batch_size

        return total_loss / total_samples

    def validate(self, val_loader):
        self.model.eval()
        total_loss = 0
        total_samples = 0

        with torch.no_grad():
            for images, maps in val_loader:
                images = images.to(self.device, non_blocking=True)
                maps = maps.to(self.device, non_blocking=True)

                with torch.autocast(device_type=self.device.type, dtype=torch.float16, enabled=self.use_amp):
                    logit = self.model(images)
                    loss = self.loss_function(logit, maps)

                batch_size = images.size(0)
                total_loss += (loss.detach().float().item() * batch_size)
                total_samples += batch_size

        return total_loss / total_samples

    def fit(self, train_loader, val_loader, epochs, trial=None):

        history = {"train_loss": [], "val_loss": []}
        best_val_loss = float("inf")
        best_model_state = None
        best_epoch = None

        for epoch in range(epochs):
            epoch_start = time.perf_counter()

            train_loss = self.train_epoch(train_loader)
            val_loss = self.validate(val_loader)
            
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)

            #best model
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_epoch = epoch + 1
                best_model_state = {key: value.detach().cpu().clone() for key, value in self.model.state_dict().items()}
            epoch_time = time.perf_counter() - epoch_start
            print(f"epoch {epoch + 1}/{epochs} | train BCE: {train_loss:.6f} | val BCE: {val_loss:.6f} | best loss: {best_val_loss:.6f} | (epoch {best_epoch}) | time: {epoch_time:.1f}s", flush=True)
            
            if trial is not None:
                trial.report(val_loss, step=epoch)
                if trial.should_prune():
                    print(f"trial {trial.number} PRUNED | epoch={epoch + 1} | val_BCE={val_loss:.6f}", flush=True)
                    raise optuna.TrialPruned()

        if best_model_state is not None:
            self.model.load_state_dict(best_model_state)
        
        return best_val_loss, history

In [ ]:
def objective(trial, pretrained=False, frozen=False):
    """ Optuna objective. It runs an optuna trial and returns the best validation loss. """
    
    #same seed for everything in order to maximize reproducibility
    seed = 42

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


    # hyperparameters space
    batch_size = trial.suggest_categorical("batch_size",[4, 8, 16, 32])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "AdamW", "RMSprop", "NAdam", "SGD"])
    weight_decay = trial.suggest_float("weight_decay", 1e-8, 1e-3, log=True)

    # loss
    loss_function = nn.BCEWithLogitsLoss()

    # decoder lr
    if optimizer_name == "SGD":
        lr_decoder = trial.suggest_float("lr_decoder", 1e-4, 1e-1, log=True)
    else:
        lr_decoder = trial.suggest_float("lr_decoder", 1e-5, 1e-2, log=True)

    #encoder lr if not frozen
    if not frozen:
        #fine tuning
        if pretrained:
            if optimizer_name == "SGD":
                lr_encoder = trial.suggest_float("lr_encoder", 1e-5, 1e-2, log=True)
            else:
                lr_encoder = trial.suggest_float("lr_encoder", 1e-6, 1e-4, log=True)
        #coarser tuning
        else:
            if optimizer_name == "SGD":
                lr_encoder = trial.suggest_float("lr_encoder", 1e-4, 1e-1, log=True)
            else:
                lr_encoder = trial.suggest_float("lr_encoder", 1e-5, 1e-2, log=True)

    #data loader
    loader_options = dict(num_workers=12, pin_memory=(device.type == "cuda"), persistent_workers=True, prefetch_factor=4)
    train_loader = DataLoader(training_set, batch_size=batch_size, shuffle=True, **loader_options)
    val_loader = DataLoader(validation_set, batch_size=batch_size, shuffle=False, **loader_options)

    #model
    model = encoder_decoder(pretrained=pretrained, frozen=frozen).to(device)

    #parameter groups
    if frozen:
        # frozen encoder, excluded from optimizer
        parameter_groups = [{"params": model.decoder.parameters(), "lr": lr_decoder}]
    else:
        parameter_groups = [{"params": model.encoder.parameters(), "lr": lr_encoder}, 
                            {"params": model.decoder.parameters(), "lr": lr_decoder}]
        
    optimizer_options = {"weight_decay": weight_decay}

    if optimizer_name in ["SGD", "RMSprop"]:
        optimizer_options["momentum"] = trial.suggest_float("momentum", 0.0, 0.95)
    else:
        beta1 = trial.suggest_float("beta1", 0.8, 0.99)
        beta2 = trial.suggest_float("beta2", 0.9, 0.9999)
        optimizer_options["betas"] = (beta1, beta2)

    # history log
    print("\n" + "=" * 84, flush=True)
    print(f"trial {trial.number}", flush=True)
    print(f"pretrained={pretrained} | frozen={frozen} | optimizer={optimizer_name} | batch={batch_size}",flush=True)
    
    if frozen:
        print(f"lr_decoder={lr_decoder:.2e} | weight_decay={weight_decay:.2e}", flush=True)
    else:
        print(f"lr_encoder={lr_encoder:.2e} | lr_decoder={lr_decoder:.2e} | weight_decay={weight_decay:.2e}", flush=True)

    print(f"optimizer_options={optimizer_options}", flush=True)
    print("=" * 84, flush=True)

    #converting optuna optimizer name into pytorch class, e.g. Adam -> torch.optim.Adam
    optimizer_class = getattr(torch.optim, optimizer_name)
    optimizer = optimizer_class(parameter_groups, **optimizer_options)

    model_trainer = trainer(model=model, optimizer=optimizer,loss_function=loss_function, device=device)

    best_val_loss, history = model_trainer.fit(train_loader=train_loader, val_loader=val_loader, epochs=30, trial=trial)

    print(f"\ntrial {trial.number} COMPLETE | best loss: {best_val_loss:.6f}",flush=True)

    return best_val_loss

In [ ]:
#to optune de model, set pretrained and frozen configuration and change the study name and the storage, then run .py and hope everything works fine. 

study = optuna.create_study(study_name="MLResnet18_pretrained_frozen_BCEloss", storage="sqlite:///optuna_studies/optuna_MLresnet18_pretrained_frozen_BCEloss.db", 
                            direction="minimize", load_if_exists=True, pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5))
study.optimize(lambda trial: objective(trial, pretrained=True, frozen=True ), n_trials=25)